# Graph data validation and features

## Goal

This walkthrough uses synthetic node and edge tables to demonstrate directed graph semantics, duplicate and dangling-edge detection, validation, degree features, connected components, and adjacency conversion.

In [1]:
import pandas as pd
from autoprepml import GraphPrepML

nodes = pd.DataFrame({"node_id": [1, 2, 3, 4, 4, 5], "label": ["A", "B", "C", "D", "duplicate", "E"]})
edges = pd.DataFrame({
    "source_id": [1, 1, 2, 3, 4, 4, 9, 5],
    "target_id": [2, 3, 3, 4, 5, 5, 1, 5],
    "weight": [1.0, 2.0, 1.5, 0.5, 3.0, 3.0, 1.0, 1.0],
})
nodes

,node_id,label
0,1,A
1,2,B
2,3,C
3,4,D
4,4,duplicate
5,5,E


### Detect and clean graph integrity issues

In [2]:
preparer = GraphPrepML(
    nodes_df=nodes, edges_df=edges, node_id_col="node_id",
    source_col="source_id", target_col="target_id", directed=True
)
issues = preparer.detect_issues()
print("node_issues:", issues["nodes"])
print("edge_issues:", issues["edges"])
preparer.validate_node_ids()
preparer.validate_edges(remove_self_loops=True, remove_dangling=True)
preparer.remove_duplicate_edges()
print("clean_shapes:", preparer.nodes_df.shape, preparer.edges_df.shape)

node_issues: {'total_nodes': 6, 'duplicate_node_ids': 1, 'missing_node_ids': 0}
edge_issues: {'total_edges': 8, 'duplicate_edges': 1, 'self_loops': 1, 'missing_sources': 0, 'missing_targets': 0, 'dangling_edges': 1}
clean_shapes: (5, 2) (5, 3)


### Add graph features and representations

In [3]:
preparer.add_node_features()
preparer.add_edge_features()
preparer.identify_components()
stats = preparer.get_graph_stats()
print("graph_stats:", stats)
print("adjacency:", preparer.to_adjacency_dict())
preparer.nodes_df[["node_id", "out_degree", "in_degree", "total_degree", "component_id"]]

graph_stats: {'directed': True, 'num_nodes': 5, 'num_edges': 5, 'density': 0.25, 'avg_degree': 2.0}
adjacency: {1.0: [2.0, 3.0], 2.0: [3.0], 3.0: [4.0], 4.0: [5.0]}


,node_id,out_degree,in_degree,total_degree,component_id
0,1,2,0,2,0
1,2,1,1,2,0
2,3,1,2,3,0
3,4,1,1,2,0
5,5,0,1,1,0


## Checks

In [4]:
assert preparer.nodes_df["node_id"].is_unique
assert set(preparer.edges_df["source_id"]).issubset(set(preparer.nodes_df["node_id"]))
assert set(preparer.edges_df["target_id"]).issubset(set(preparer.nodes_df["node_id"]))
assert "component_id" in preparer.nodes_df
print("Graph workflow checks passed.")

Graph workflow checks passed.
